# BDSS Dev Cluster - quick start
Spark on YARN reading a file from the bind-mounted `/data` folder, then writing to HDFS.

_Built by Mohamed ([fogknight-10/bdss-dev-cluster](https://github.com/fogknight-10/bdss-dev-cluster)) using the original cluster by usf-etman._

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("bdss-quickstart")
         .master("yarn")                      # the real cluster; use local[*] for quick tests
         .getOrCreate())
print(spark.version, "| master:", spark.sparkContext.master)

## 1. Create a file in `/data`
This folder is `data/` on your computer. Anything saved here is visible to Spark on every worker - no copying.

In [ ]:
import csv, os

os.makedirs("/data/quickstart", exist_ok=True)
with open("/data/quickstart/orders.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    for i in range(10000):
        w.writerow([i % 100, 1000 + i % 37, round(1 + (i * 7919) % 5000 / 100, 2)])
print("written: data/quickstart/orders.csv")

## 2. Read it with Spark on YARN
`file:///` makes the executors read the bind mount, which every worker has at the same path.

In [ ]:
orders = spark.read.csv("file:///data/quickstart/orders.csv").toDF("customer", "item", "amount")
orders.show(5)
print(orders.count(), "rows")

## 3. Write to HDFS and read it back
A path starting with `hdfs:///` lives in the cluster (a bare `/path` also means HDFS when HDFS is the default filesystem - `verify-cluster.sh` prints it).

In [ ]:
orders.write.mode("overwrite").parquet("hdfs:///user/quickstart/orders")
spark.read.parquet("hdfs:///user/quickstart/orders").groupBy("customer").count().orderBy("count", ascending=False).show(5)

## 4. Copy a folder into HDFS (optional)
In a terminal: `bash /opt/bdss-scripts/hdfs-sync.sh /data/quickstart /data/quickstart`, then read `hdfs:///data/quickstart/orders.csv`.

In [ ]:
spark.stop()